In [ ]:
# 与源目录结构保持一致，争对aiops代码
'''
datasets/
└── train/
    └── groundtruth/
        └── groundtruth_2022-05-01.json
        └── ...
'''

In [1]:
import os
import re
import json
import pickle
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd


# 你原来的中文->5类映射可以保留
# ANOMALY_DICT = {
#     "k8s容器网络资源包重复发送": "network",
#     "k8s容器写io负载": "io",
#     "k8s容器读io负载": "io",
#     "k8s容器cpu负载": "cpu",
#     "k8s容器网络延迟": "network",
#     "k8s容器进程中止": "process",
#     "k8s容器网络丢包": "network",
#     "k8s容器内存负载": "memory",
#     "k8s容器网络资源包损坏": "network",

#     "node节点CPU故障": "cpu",
#     "node 磁盘读IO消耗": "io",
#     "node 内存消耗": "memory",
#     "node 磁盘空间消耗": "memory",
#     "node 磁盘写IO消耗": "io",
#     "node节点CPU爬升": "cpu",
# }

FAULT_TYPE_MAP_15 = {
    # ---- node faults (前 6 个) ----
    "node节点CPU故障": "node_cpu_failure",
    "node 磁盘读IO消耗": "node_disk_read_io_load",
    "node 内存消耗": "node_memory_load",
    "node 磁盘空间消耗": "node_disk_space_load",
    "node 磁盘写IO消耗": "node_disk_write_io_load",
    "node节点CPU爬升": "node_cpu_spike",

    # ---- service / pod faults (后 9 个) ----
    "k8s容器网络资源包重复发送": "container_network_packet_duplicate",
    "k8s容器写io负载": "container_write_io_load",
    "k8s容器读io负载": "container_read_io_load",
    "k8s容器cpu负载": "container_cpu_load",
    "k8s容器网络延迟": "container_network_latency",
    "k8s容器进程中止": "container_process_abort",
    "k8s容器网络丢包": "container_network_packet_loss",
    "k8s容器内存负载": "container_memory_load",
    "k8s容器网络资源包损坏": "container_network_packet_corruption",
}

FAULT_TYPE_LIST_15 = [
    # node 6
    "node_cpu_failure",
    "node_disk_read_io_load",
    "node_memory_load",
    "node_disk_space_load",
    "node_disk_write_io_load",
    "node_cpu_spike",

    # service / pod 9
    "container_network_packet_duplicate",
    "container_write_io_load",
    "container_read_io_load",
    "container_cpu_load",
    "container_network_latency",
    "container_process_abort",
    "container_network_packet_loss",
    "container_memory_load",
    "container_network_packet_corruption",
]


DEFAULT_POD_SUFFIXES = ["-0", "-1", "-2", "2-0"]


def ensure_dir(p: str) -> str:
    os.makedirs(p, exist_ok=True)
    return p


def parse_date_from_any_name(name: str) -> Optional[str]:
    m = re.search(r"(20\d{2}-\d{2}-\d{2})", name)
    return m.group(1) if m else None


def parse_cloudbed_from_train_csv_name(fname: str) -> Optional[str]:
    m = re.search(r"k8s-(\d+)", fname)
    return f"cloudbed-{m.group(1)}" if m else None


class TimeIntervalLabelGeneratorAlignedToOpenSource:
    """
    目标：复刻你上传的源代码 TimeIntervalLabelGenerator 的行为：
    - 遍历 groundtruth 的每条故障记录
    - 每条故障记录生成 1 个时间窗（当前开源版本 slice_ground_truth_timestamp 只返回 1 个 interval）
    - y 是 entity-level fault_type id 向量（长度 E）
    - time_interval / y 的长度：≈ groundtruth 条目数（若 slice 返回多个则按多个重复
    """

    def __init__(
        self,
        datasets_root: str,
        out_base: str,
        all_entity_list: List[str],
        train_dirname: str = "training_data_with_faults",
        test_dirname: str = "test_data",
        default_cloudbed: str = "cloudbed-1",
        date2cloudbeds: Optional[Dict[str, List[str]]] = None,
        fixed_fault_type_list: Optional[List[str]] = None,
        pod_suffixes: List[str] = DEFAULT_POD_SUFFIXES,
        sliding_ratio: float = 0.5,          # 开源里传 0.5
        skip_bad_case: bool = True,
        bad_case_date: str = "2022-03-24",
        bad_case_cloudbeds: Tuple[str, ...] = ("cloudbed-1", "cloudbed-2"),
    ):
        self.datasets_root = datasets_root
        self.out_base = out_base
        self.all_entity_list = all_entity_list
        self.name2idx = {n: i for i, n in enumerate(all_entity_list)}
        self.E = len(all_entity_list)

        self.train_dirname = train_dirname
        self.test_dirname = test_dirname
        self.default_cloudbed = default_cloudbed
        self.date2cloudbeds = date2cloudbeds
        self.pod_suffixes = pod_suffixes

        self.sliding_ratio = float(sliding_ratio)
        self.skip_bad_case = skip_bad_case
        self.bad_case_date = bad_case_date
        self.bad_case_cloudbeds = bad_case_cloudbeds

        if fixed_fault_type_list is None:
            fixed_fault_type_list = FAULT_TYPE_LIST_15
        self.fault_type_list = list(fixed_fault_type_list)
        self.ft2id = {ft: i + 1 for i, ft in enumerate(self.fault_type_list)}  # 1..K

    def _iter_files(self, folder: str, suffix: str) -> List[str]:
        if not os.path.isdir(folder):
            return []
        return sorted([os.path.join(folder, fn) for fn in os.listdir(folder) if fn.endswith(suffix)])


    # ---------- groundtruth 读取（输出格式对齐源码期望字段） ----------
    def load_train_groundtruth_list(self) -> Dict[str, Dict[str, List[dict]]]:
        """
        输出结构：
          out[date][cloudbed] = [
            {
              "timestamp": int,
              "entity_type": "pod"/"service"/"node",
              "cmdb_id": str,
              "fault_type": int,          # 1..K
              "template": str,            # 源码里会保存 template
              "root_cause_type": str      # 源码里会保存 root_cause_type
            }, ...
          ]
        """
        gt_dir = os.path.join(self.datasets_root, self.train_dirname, "groundtruth")
        files = self._iter_files(gt_dir, ".csv")
        out: Dict[str, Dict[str, List[dict]]] = {}

        for fpath in files:            
            fname = os.path.basename(fpath)
            date_str = parse_date_from_any_name(fname)
            cloudbed = parse_cloudbed_from_train_csv_name(fname) or self.default_cloudbed
            if not date_str:
                continue

            df = pd.read_csv(fpath)
            need_cols = {"timestamp", "level", "cmdb_id", "failure_type"}
            if not need_cols.issubset(set(df.columns)):
                raise ValueError(f"[train csv] missing columns in {fpath}, got={df.columns.tolist()}")

            items: List[dict] = []
            for _, r in df.iterrows():
                level = str(r["level"]).lower().strip()  # node/service/pod
                cmdb_id = str(r["cmdb_id"]).strip()
                raw_type = str(r["failure_type"]).strip()

                mapped = FAULT_TYPE_MAP_15.get(raw_type, raw_type)
                fault_type_id = self.ft2id.get(mapped, 0)  # 不在5类里就给 0（等价于“无法标注”）

                items.append({
                    "timestamp": int(r["timestamp"]),
                    "entity_type": level,
                    "cmdb_id": cmdb_id,
                    "fault_type": int(fault_type_id),
                    "template": raw_type,          # 用原始中文做 template（更贴近“模板/描述”字段用途）
                    "root_cause_type": mapped,     # 用5类做 root_cause_type
                })

            out.setdefault(date_str, {}).setdefault(cloudbed, []).extend(items)

        return out

    def load_test_groundtruth_list(self) -> Dict[str, Dict[str, List[dict]]]:
        gt_dir = os.path.join(self.datasets_root, self.test_dirname, "groundtruth")
        files = self._iter_files(gt_dir, ".json")
        out: Dict[str, Dict[str, List[dict]]] = {}

        for fpath in files:
            fname = os.path.basename(fpath)
            date_str = parse_date_from_any_name(fname)
            if not date_str:
                continue

            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)

            ts_list = data.get("timestamp", [])
            lv_list = data.get("level", [])
            id_list = data.get("cmdb_id", [])
            ft_list = data.get("failure_type", [])
            n = min(len(ts_list), len(lv_list), len(id_list), len(ft_list))

            cloudbeds = (self.date2cloudbeds.get(date_str, [self.default_cloudbed])
                         if self.date2cloudbeds else [self.default_cloudbed])

            for cloudbed in cloudbeds:
                items: List[dict] = []
                for i in range(n):
                    level = str(lv_list[i]).lower().strip()
                    cmdb_id = str(id_list[i]).strip()
                    raw_type = str(ft_list[i]).strip()

                    mapped = FAULT_TYPE_MAP_15.get(raw_type, raw_type)
                    fault_type_id = self.ft2id.get(mapped, 0)

                    items.append({
                        "timestamp": int(ts_list[i]),
                        "entity_type": level,
                        "cmdb_id": cmdb_id,
                        "fault_type": int(fault_type_id),
                        "template": raw_type,
                        "root_cause_type": mapped,
                    })
                out.setdefault(date_str, {}).setdefault(cloudbed, []).extend(items)

        return out

    # ---------- entity-level label vec（对齐开源：service 同时标 4 个 pod） ----------
    def get_ground_truth_label_vec(self, entity_type: str, cmdb_id: str, fault_type_id: int) -> np.ndarray:
        """
        对齐你上传的源码 get_ground_truth_label：
          label_list = [0]*E
          if cmdb_id in ent: label[idx]=fault_type
          if entity_type == 'service': 给4个pod也标 fault_type
        """
        y = np.zeros((self.E,), dtype=np.int64)

        if fault_type_id <= 0:
            return y

        # 先标记 cmdb_id 本体（如果存在于 all_entity_list）
        idx = self.name2idx.get(cmdb_id, None)
        if idx is not None:
            y[idx] = int(fault_type_id)

        # service 额外把4个 pod 也标记
        if entity_type == "service":
            for suf in self.pod_suffixes:
                pod = f"{cmdb_id}{suf}"
                pidx = self.name2idx.get(pod, None)
                if pidx is not None:
                    y[pidx] = int(fault_type_id)

        return y


    # ---------- slice_ground_truth_timestamp ----------

    @staticmethod
    def _date_start_ts(date: str) -> int:
        # 秒级 00:00:00
        return int(pd.Timestamp(f"{date} 00:00:00").timestamp())

    def slice_ground_truth_timestamp(
        self,
        date: str,
        cloudbed: str,
        ground_truth_timestamp: int,
        window_size_min: int,
    ) -> List[Tuple[str, str, int, int]]:
        """
        对齐开源实现（当前版本只返回 1 个 interval）：
          start_timestamp = date 00:00:00
          c_ts = gt_ts - (gt_ts-start_ts)%60   # 以“当天起点”为基准向下取整到分钟
          s_ts = c_ts - int(window_size*sliding_ratio)*60
          e_ts = s_ts + window_size*60
        """
        # 拿到当天的开始时间戳
        start_ts = self._date_start_ts(date)
        # 故障时间戳
        gt_ts = int(ground_truth_timestamp)

        # 故障时间戳计算分钟时间桶
        c_ts = gt_ts - ((gt_ts - start_ts) % 60)
        # 启始
        s_ts = c_ts - int(window_size_min * self.sliding_ratio) * 60
        # 终止
        e_ts = s_ts + int(window_size_min) * 60
        #  ("2022-03-20", "cloudbed-1", 1647734460, 1647735060)
        return [(date, cloudbed, int(s_ts), int(e_ts))]

    
    
     # ---------- ✅ 生成 time_interval + y ----------
    def generate_time_interval_label(self, window_size_min: int) -> Dict[str, Any]:
        train_gt = self.load_train_groundtruth_list()
        test_gt = self.load_test_groundtruth_list()

        
        result = {
            "time_interval": {"train_valid": [], "test": []},
            "y": {"train_valid": [], "test": []},  # 每个元素 shape=(E,)
            "entity_type": {"train_valid": [], "test": []},
            "template": {"train_valid": [], "test": []},
            "cmdb_id": {"train_valid": [], "test": []},
            "root_cause_type": {"train_valid": [], "test": []},
        }

        split2dtype = {"train_valid": self.train_dirname, "test": self.test_dirname}
        split2gt = {"train_valid": train_gt, "test": test_gt}


        for split_key in ["train_valid", "test"]:
            gt_dict = split2gt[split_key]

            # 遍历这个 split 的所有 (date, cloudbed) groundtruth
            for date, cb_dict in gt_dict.items():
                for cloudbed, items in cb_dict.items():
                    if self.skip_bad_case and date == self.bad_case_date and cloudbed in self.bad_case_cloudbeds:
                        continue

                    # 按时间排序逐条生成 interval
                    items = sorted(items, key=lambda x: int(x["timestamp"]))
                    for it in items:
                        ft_id = int(it["fault_type"])
                        if ft_id <= 0:
                            continue
    
                        lv = str(it["entity_type"]).lower().strip()
                        cid = str(it["cmdb_id"]).strip()
                        tmp = str(it.get("template", ""))
                        rc = str(it.get("root_cause_type", ""))
    
                        intervals = self.slice_ground_truth_timestamp(
                            date=date,
                            cloudbed=cloudbed,
                            ground_truth_timestamp=int(it["timestamp"]),
                            window_size_min=int(window_size_min),
                        )
    
                        # time_interval 扩展
                        result["time_interval"][split_key].extend([list(x) for x in intervals])
    
                        # y：interval 有几个就重复几个（开源就是这么 extend 的） :contentReference[oaicite:4]{index=4}
                        y_vec = self.get_ground_truth_label_vec(lv, cid, ft_id)
                        result["y"][split_key].extend([y_vec for _ in range(len(intervals))])
    
                        # extra 字段：开源代码是每条 groundtruth append 一次（依赖 intervals 长度=1） :contentReference[oaicite:5]{index=5}
                        result["entity_type"][split_key].append(lv)
                        # 注意：开源的 template 存的是“去掉 pod 后缀的 cmdb_id”（不是 failure_type 文本） :contentReference[oaicite:6]{index=6}
                        # 你要严格对齐开源的话，建议用下面这一行：
                        result["template"][split_key].append(
                            cid.replace('2-0', '').replace('-0', '').replace('-1', '').replace('-2', '').replace('-3', '')
                               .replace('-4', '').replace('-5', '').replace('-6', '')
                        )
                        result["cmdb_id"][split_key].append(cid)
                        result["root_cause_type"][split_key].append(rc)

        return result

    def dump(self, window_size_list: List[int]):
        out_dir = ensure_dir(os.path.join(self.out_base, "dataset", "time_interval_and_label"))
        for w in window_size_list:
            obj = self.generate_time_interval_label(int(w))
            out_pkl = os.path.join(out_dir, f"time_interval_window_size_{w}.pkl")

            '''
                {
                  "time_interval": {
                      "train_valid": [...],
                      "test": [...]
                  },
                
                  "y": {
                      "train_valid": [...],
                      "test": [...]
                  },
                
                  "entity_type": {
                      "train_valid": [...],
                      "test": [...]
                  },
                
                  "template": {
                      "train_valid": [...],
                      "test": [...]
                  },
                
                  "cmdb_id": {
                      "train_valid": [...],
                      "test": [...]
                  },
                
                  "root_cause_type": {
                      "train_valid": [...],
                      "test": [...]
                  }
                }
                ====================================================
                time_interval是一个列表
                [
                 ("2022-03-20","cloudbed-1",1647763200,1647763800),
                 ("2022-03-20","cloudbed-1",1647764100,1647764700),
                ]
                ====================================================
                y 是元素标签列表
                [
                    np.array(E,),
                    np.array(E,),
                    ...
                ]
                每个元素array([0,0,0,2,0,0,0,0,...])
                ====================================================
                entity_type为故障实体类型
                ["service","pod","node","service",...]
                ====================================================
                template为service名
                ["checkoutservice","frontend",...]
                ====================================================
                cmdb_id为发生故障的实体
                ["checkoutservice","node-3",...]
                ====================================================
                root_cause_type为根因类型
                ["network","cpu","io",...]
                ====================================================
                所有数据长度都相等
            '''
            
            with open(out_pkl, "wb") as f:
                pickle.dump(obj, f)
            print(f"[OK] saved -> {out_pkl} "
                  f"train_valid_intervals={len(obj['time_interval']['train_valid'])}, "
                  f"test_intervals={len(obj['time_interval']['test'])}")

In [2]:
all_entity_list = [
    'node-1', 
    'node-2', 
    'node-3', 
    'node-4', 
    'node-5', 
    'node-6', 
    'adservice', 
    'cartservice', 
    'checkoutservice', 
    'currencyservice', 
    'emailservice', 
    'frontend', 
    'paymentservice', 
    'productcatalogservice', 
    'recommendationservice', 
    'shippingservice', 
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [3]:
gen = TimeIntervalLabelGeneratorAlignedToOpenSource(
    datasets_root="/home/ZZData/Eadro/preprocess/datasets/aiops2022/",
    out_base="../../temp_data/aiops22",
    all_entity_list=all_entity_list,
    train_dirname="training_data_with_faults",
    test_dirname="test_data",
    default_cloudbed="cloudbed",
    fixed_fault_type_list=FAULT_TYPE_LIST_15,
    sliding_ratio=0.5,        # 对齐开源
    skip_bad_case=True,
)

# 会输出：
# ../../temp_data/aiops22/dataset/time_interval_and_label/time_interval_window_size_{w}.pkl
gen.dump([5, 10, 15])

[OK] saved -> ../../temp_data/aiops22/dataset/time_interval_and_label/time_interval_window_size_5.pkl train_valid_intervals=300, test_intervals=241
[OK] saved -> ../../temp_data/aiops22/dataset/time_interval_and_label/time_interval_window_size_10.pkl train_valid_intervals=300, test_intervals=241
[OK] saved -> ../../temp_data/aiops22/dataset/time_interval_and_label/time_interval_window_size_15.pkl train_valid_intervals=300, test_intervals=241
